# LC #1 — Two Sum
**Category:** HashMap | **Difficulty:** Easy | **Day 1**

---

<div style="padding:15px;border-left:8px solid #667eea;background:#f0f0ff;border-radius:4px;">
<strong>The Problem:</strong> Given an integer array <code>nums</code> and an integer <code>target</code>,
return the indices of two numbers that add up to <code>target</code>.
Exactly one solution exists. Each element may only be used once.
</div>

**Example:**
```
Input:  nums = [2, 7, 11, 15], target = 9
Output: [0, 1]   (nums[0] + nums[1] = 2 + 7 = 9)

Input:  nums = [3, 2, 4], target = 6
Output: [1, 2]
```

**Core Insight:** For each number `x`, you need `target - x`. Instead of scanning for it (O(n) per lookup = O(n²) total), store every number you've seen in a hashmap. Then each lookup is O(1).

## Mental Models

**1. The Complement Store**
Every number has a "partner" it needs: `partner = target - num`. Before checking the current number, ask: "Is my partner already in the store?" If yes — done. If no — add yourself to the store and keep going.

**2. One-Pass Building**
You don't need to build the entire hashmap first and then search. Build and search simultaneously. The moment you find a pair, return. This makes it genuinely O(n) — not O(n) + O(n).

**3. Space for Time Tradeoff**
The classic optimization pattern: spend O(n) extra space (the hashmap) to buy O(n) total time instead of O(n²). At scale — 6,000 servers, millions of data points — this difference is enormous.

In [ ]:
# Brute Force — O(n²) time, O(1) space
# Check every pair. Simple but slow.

def twoSum_brute(nums, target):
    for i in range(len(nums)):
        for j in range(i + 1, len(nums)):
            if nums[i] + nums[j] == target:
                return [i, j]
    # n*(n-1)/2 comparisons — 1000 elements = ~500,000 checks

# Test
print(twoSum_brute([2, 7, 11, 15], 9))   # [0, 1]
print(twoSum_brute([3, 2, 4], 6))         # [1, 2]
print(twoSum_brute([3, 3], 6))            # [0, 1]

## Step-by-Step: HashMap Approach

Trace through `nums = [2, 7, 11, 15]`, `target = 9`:

| Step | `i` | `num` | `complement` | In `seen`? | Action |
|------|-----|-------|-------------|-----------|--------|
| 1 | 0 | 2 | 7 | No | `seen = {2: 0}` |
| 2 | 1 | 7 | 2 | **Yes** | Return `[seen[2], 1]` = `[0, 1]` ✓ |

The key insight: at step 2, we check "is 2 in seen?" before adding 7. This means we find the pair in one pass, even if the earlier element came first.

In [ ]:
# Optimal — O(n) time, O(n) space
# One pass: check complement, then store current number.

def twoSum(nums: list[int], target: int) -> list[int]:
    seen = {}  # maps value → index

    for i, num in enumerate(nums):
        complement = target - num
        if complement in seen:
            return [seen[complement], i]
        seen[num] = i

    # Guaranteed to find a solution (problem states exactly one exists)

# Test
print(twoSum([2, 7, 11, 15], 9))   # [0, 1]
print(twoSum([3, 2, 4], 6))         # [1, 2]
print(twoSum([3, 3], 6))            # [0, 1]  ← works: checks seen before adding self

## Complexity Analysis

| Approach | Time | Space | Notes |
|----------|------|-------|-------|
| Brute Force (two loops) | O(n²) | O(1) | n*(n-1)/2 comparisons |
| Sort + Two Pointers | O(n log n) | O(n) | Loses original indices |
| **HashMap (Optimal)** | **O(n)** | **O(n)** | One pass, O(1) lookup |

**Why O(1) lookup?**
Python `dict` is a hash table. Average case O(1) for `in` and `[]` operations. Worst case O(n) with pathological hash collisions — doesn't happen with integers in practice.

**Space note:** We store at most n entries in the hashmap — one per element. O(n) space.

## Interview Q&A

**Q: Why check complement before adding to the map?**
A: Prevents using the same element twice. If `nums = [3, 3]` and target = 6: when we process the second 3, the first 3 is already in seen at index 0. We return [0, 1]. If we added first and then checked, we'd look up index 0 for both — but the problem says same index can't be used twice.

**Q: What's `seen` storing — keys or values?**
A: Keys are the **values** from `nums`, values in the dict are the **indices**. `seen = {value: index}`. This is the reverse of what you might expect — we look up by number to get its position.

**Q: What if the array had multiple valid pairs?**
A: Return on the first found. To return all pairs: collect into a list instead, skip returning early.

**Q: What data structure is Python `{}` (dict) under the hood?**
A: A hash table. The CPython implementation uses open addressing with pseudo-random probing. Average O(1) for insert/lookup/delete. Guaranteed O(n) worst case (very rare with integers).

**Q: Could you solve this with O(1) space?**
A: Yes — sort a copy (keeping original indices), then use two pointers. But sorting loses index info, so you'd need to track original indices separately. And it's O(n log n) time, not O(n).

## The Citi Angle

**Pattern recognition:** At Citi, monitoring 6,000+ endpoints, I frequently needed to find pairs or groups of resources with complementary behavior — servers where one's CPU spike was offset by another's idle period (capacity balancing).

**Direct application:** If you have a list of server CPU readings and want to find two servers whose combined utilization equals exactly 100% (for load-balancing pairing):
```python
cpu_readings = {server_id: cpu_pct for server_id in servers}
# Two Sum pattern: complement = 100 - cpu_pct
```

**Interview tie-in:** "The HashMap pattern is foundational to data engineering. At Citi, fast lookup against growing datasets was constant — whether matching alert IDs, deduplicating metric keys, or joining telemetry streams. O(1) lookup vs O(n) lookup on 6,000 servers per collection cycle is the difference between milliseconds and seconds."

## Summary

| | Value |
|--|--|
| **Pattern** | HashMap — complement lookup |
| **Time** | O(n) |
| **Space** | O(n) |
| **Key line** | `if complement in seen: return [seen[complement], i]` |
| **Say in interview** | "This is a complement-store problem. O(n) time by trading O(n) space for O(1) lookup." |

**The one-liner to memorize:**
```python
seen = {}
for i, num in enumerate(nums):
    if target - num in seen: return [seen[target-num], i]
    seen[num] = i
```